In [2]:
!pip install spacy
!python -m spacy download en_core_web_sm

import re
import spacy
import json
from spacy import displacy

print("Setup complete")

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Setup complete


# 2.1 Extract Dates

In [3]:
def extract_dates(text):
    # Pattern: DD/MM/YYYY, YYYY-MM-DD, March 15, 2024, etc.
    date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}\b'
    return re.findall(date_pattern, text, re.IGNORECASE)

# Test
sample = "Today is 15/03/2024 and deadline: March 20, 2025"
print(extract_dates(sample))

['15/03/2024', 'March 20, 2025']


# 2.2 Extract Currency Amounts

In [4]:
def extract_amounts(text):
    pattern = r'\$\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?|\d{1,3}(?:,\d{3})*(?:\.\d{2})?\s?(?:USD|dollars)?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amt in amounts:
        clean = amt.replace('$', '').replace(',', '').strip()
        # extract first number if string has extra words
        num_match = re.search(r'\d+(?:\.\d+)?', clean)
        if num_match:
            cleaned.append(float(num_match.group()))
    return cleaned

# Test
test_text = "Total: $1,250.50, Tax: 125.05 USD, Subtotal: 1125.45"
print(extract_amounts(test_text))

[1250.5, 125.05, 112.0, 5.45]


# 2.3 Extract Invoice/Order Numbers

In [5]:
def extract_invoice_number(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#?\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice(?:\s+Number|\s+#)?\s*:?\s*([A-Z0-9\-]+)'
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# Test
print(extract_invoice_number("Invoice #: INV-2024-001"))

INV-2024-001


In [6]:
def extract_dates(text):
    # Pattern: DD/MM/YYYY, YYYY-MM-DD, March 15, 2024, etc.
    date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}\b'
    return re.findall(date_pattern, text, re.IGNORECASE)

# Test
sample = "Today is 15/03/2024 and deadline: March 20, 2025"
print(extract_dates(sample))
def extract_amounts(text):
    pattern = r'\$\s?\d{1,3}(?:,\d{3})*(?:\.\d{2})?|\d{1,3}(?:,\d{3})*(?:\.\d{2})?\s?(?:USD|dollars)?'
    amounts = re.findall(pattern, text)
    cleaned = []
    for amt in amounts:
        clean = amt.replace('$', '').replace(',', '').strip()
        # extract first number if string has extra words
        num_match = re.search(r'\d+(?:\.\d+)?', clean)
        if num_match:
            cleaned.append(float(num_match.group()))
    return cleaned

# Test
test_text = "Total: $1,250.50, Tax: 125.05 USD, Subtotal: 1125.45"
print(extract_amounts(test_text))
def extract_invoice_number(text):
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#?\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice(?:\s+Number|\s+#)?\s*:?\s*([A-Z0-9\-]+)'
    ]
    for pat in patterns:
        match = re.search(pat, text, re.IGNORECASE)
        if match:
            return match.group(1) if match.groups() else match.group(0)
    return None

# Test
print(extract_invoice_number("Invoice #: INV-2024-001"))

['15/03/2024', 'March 20, 2025']
[1250.5, 125.05, 112.0, 5.45]
INV-2024-001


# 3.1 Load model and basic NER

In [7]:
nlp = spacy.load('en_core_web_sm')

invoice_text = """Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50"""

doc = nlp(invoice_text)

print("All entities:")
for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:10} {spacy.explain(ent.label_)}")

All entities:
Acme Corporation          ORG        Companies, agencies, institutions, etc.
123                       CARDINAL   Numerals that do not fall under another type
Main Street               FAC        Buildings, airports, highways, bridges, etc.
New York                  GPE        Countries, cities, states
10001                     DATE       Absolute or relative dates or periods
John Smith                PERSON     People, including fictional
March 15, 2024            DATE       Absolute or relative dates or periods
1,250.50                  MONEY      Monetary values, including unit


# 3.2 Extract specific entity types

In [8]:
def extract_entities(text):
    doc = nlp(text)
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
    return entities

result = extract_entities(invoice_text)
for k, v in result.items():
    print(f"{k}: {v}")

persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


# 3.3 Visualize with displaCy

In [10]:
# Visualize inside notebook (inline)
displacy.render(doc, style='ent', jupyter=True)

# Save as HTML file (for download/submission)
html_content = displacy.render(doc, style='ent', page=True, jupyter=False)
if html_content:
    with open('/kaggle/working/entities.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    print("✅ HTML saved to /kaggle/working/entities.html")
else:
    print("⚠️ displacy.render returned None. Check your spaCy version or doc object.")

✅ HTML saved to /kaggle/working/entities.html


# Complete pipeline + JSON output 

In [11]:
def process_invoice_document(text):
    """Full extraction pipeline"""
    data = {}
    
    # Regex extraction
    data['dates'] = extract_dates(text)
    data['amounts'] = extract_amounts(text)
    data['invoice_number'] = extract_invoice_number(text)
    
    # NER extraction
    ner_results = extract_entities(text)
    data['persons'] = ner_results['persons']
    data['organizations'] = ner_results['organizations']
    data['locations'] = ner_results['locations']
    data['ner_dates'] = ner_results['dates']
    data['money_ner'] = ner_results['money']
    
    # Combined processing info
    data['full_text'] = text[:200] + "..."  # truncate for JSON
    
    return data

# Test on sample text
sample_invoice = """
INVOICE #INV-2024-042
Date: April 5, 2024
From: TechCorp Inc., 500 Oracle Parkway, Redwood City, CA
To: John Miller, 12 Baker Street, London
Total Amount: $4,299.99
"""

full_result = process_invoice_document(sample_invoice)
print(json.dumps(full_result, indent=2))

# Save as JSON
output_path = '/kaggle/working/extracted_data.json'
with open(output_path, 'w') as f:
    json.dump(full_result, f, indent=2)
print(f"Saved to {output_path}")

{
  "dates": [
    "April 5, 2024"
  ],
  "amounts": [
    202.0,
    4.0,
    42.0,
    5.0,
    202.0,
    4.0,
    500.0,
    12.0,
    4299.99
  ],
  "invoice_number": "INV-2024-042",
  "persons": [
    "John Miller"
  ],
  "organizations": [
    "TechCorp Inc.",
    "Oracle Parkway"
  ],
  "locations": [
    "Redwood City",
    "London"
  ],
  "ner_dates": [
    "April 5, 2024"
  ],
  "money_ner": [
    "4,299.99"
  ],
  "full_text": "\nINVOICE #INV-2024-042\nDate: April 5, 2024\nFrom: TechCorp Inc., 500 Oracle Parkway, Redwood City, CA\nTo: John Miller, 12 Baker Street, London\nTotal Amount: $4,299.99\n..."
}
Saved to /kaggle/working/extracted_data.json


# Test on 3+ sample documents

In [12]:
docs = [
    "Order #12345, Date: 12-12-2024, Amount $99.99 from Amazon",
    "Invoice Number: ORD-999X, March 1, 2025, Person: Sarah Lee, $1,200",
    "Receipt: 05/05/2024, $5.50, Vendor: Starbucks"
]

for i, doc_text in enumerate(docs, 1):
    res = process_invoice_document(doc_text)
    print(f"\n--- Document {i} ---")
    print(json.dumps(res, indent=2))


--- Document 1 ---
{
  "dates": [
    "12-12-2024"
  ],
  "amounts": [
    123.0,
    45.0,
    12.0,
    12.0,
    202.0,
    4.0,
    99.99
  ],
  "invoice_number": "#12345",
  "persons": [],
  "organizations": [
    "Amazon"
  ],
  "locations": [],
  "ner_dates": [
    "12345"
  ],
  "money_ner": [
    "99.99"
  ],
  "full_text": "Order #12345, Date: 12-12-2024, Amount $99.99 from Amazon..."
}

--- Document 2 ---
{
  "dates": [
    "March 1, 2025"
  ],
  "amounts": [
    999.0,
    1.0,
    202.0,
    5.0,
    1200.0
  ],
  "invoice_number": "ORD-999X",
  "persons": [
    "Sarah Lee"
  ],
  "organizations": [],
  "locations": [],
  "ner_dates": [
    "March 1, 2025"
  ],
  "money_ner": [
    "1,200"
  ],
  "full_text": "Invoice Number: ORD-999X, March 1, 2025, Person: Sarah Lee, $1,200..."
}

--- Document 3 ---
{
  "dates": [
    "05/05/2024"
  ],
  "amounts": [
    5.0,
    5.0,
    202.0,
    4.0,
    5.5
  ],
  "invoice_number": null,
  "persons": [],
  "organizations": [],
  "l